# CPE4310 – ทำนายพฤติกรรมพนักงานจากข้อมูล RFID

โน้ตบุ๊กนี้ใช้ไฟล์ **`rfid_clean_kmeans.csv`** (ผลลัพธ์จาก `clean_rfid_kmeans.py`) วางไว้โฟลเดอร์เดียวกับโน้ตบุ๊ก

| ส่วน | งาน | โมเดล |
|---|---|---|
| A | จัดกลุ่มพนักงานตามรูปแบบการทำงาน | K-Means, Hierarchical |
| B | ทำนายว่าวันนั้นพนักงานจะเลิกงานดึกหรือไม่ (สแกนสุดท้าย ≥ 19:00) | Decision Tree, Random Forest, KNN, Naive Bayes, SVM, ANN |
| C | ทำนายว่าพรุ่งนี้พนักงานจะไม่มาทำงานหรือไม่ | โมเดลชุดเดียวกับ B |

ทุกโมเดลในส่วน B และ C ใช้เฉพาะข้อมูล **ก่อนวันที่ทำนาย** เป็นฟีเจอร์ และแบ่งเทรน/ทดสอบตามเวลา (75% แรกเทรน, 25% หลังทดสอบ) เพื่อไม่ให้ผลดูดีเกินจริง

In [ ]:
# รันครั้งเดียวเพื่อติดตั้งไลบรารี (ชื่อแพ็กเกจคือ scikit-learn ไม่ใช่ sklearn)
%pip install -q scikit-learn pandas numpy matplotlib

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager

from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import (silhouette_score, accuracy_score, f1_score,
                             roc_auc_score, average_precision_score,
                             confusion_matrix, ConfusionMatrixDisplay)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

RANDOM_STATE = 42
DATA = "rfid_clean_kmeans.csv"
LATE_HOUR = 19        # ส่วน B: สแกนสุดท้ายตั้งแต่เวลานี้ (ชม.) ถือว่าเลิกงานดึก
TRAIN_FRAC = 0.75     # สัดส่วนวันที่ใช้เทรน (เรียงตามเวลา)

# ฟอนต์ไทยสำหรับกราฟ (Windows มี Tahoma / Leelawadee UI อยู่แล้ว)
installed = {f.name for f in font_manager.fontManager.ttflist}
for name in ["Tahoma", "Leelawadee UI", "Noto Sans Thai", "TH Sarabun New"]:
    if name in installed:
        plt.rcParams["font.family"] = name
        break
plt.rcParams["axes.unicode_minus"] = False

## 1) โหลดข้อมูล

In [ ]:
daily = pd.read_csv(DATA, parse_dates=["date"])

def hhmm_to_hour(s):
    """'06:44' -> 6.73 (ชั่วโมงทศนิยม)"""
    if pd.isna(s) or s == "":
        return np.nan
    h, m = str(s).split(":")
    return int(h) + int(m) / 60

daily["start_h"] = daily["start_time"].apply(hhmm_to_hour)
daily["end_h"] = daily["end_time"].apply(hhmm_to_hour)
daily = daily.sort_values(["uid", "date"]).reset_index(drop=True)

ALL_DATES = np.sort(daily["date"].unique())    # วันที่ที่มีข้อมูลในระบบ (ไม่รวมวันที่ระบบไม่มีข้อมูลเลย)
print(f"{len(daily)} แถว | พนักงาน {daily['uid'].nunique()} คน | {len(ALL_DATES)} วัน "
      f"({pd.Timestamp(ALL_DATES[0]).date()} ถึง {pd.Timestamp(ALL_DATES[-1]).date()})")
daily.head()

## ส่วน A – จัดกลุ่มพนักงาน (Clustering)

สร้างฟีเจอร์ระดับพนักงาน 1 แถวต่อ 1 คน แล้วเลือกจำนวนกลุ่ม k ด้วย Silhouette Score

In [ ]:
emp = daily.groupby("name").agg(
    days=("date", "nunique"),
    first_date=("date", "min"),
    last_date=("date", "max"),
    start_mean=("start_h", "mean"),
    start_std=("start_h", "std"),
    end_mean=("end_h", "mean"),
    scans_mean=("n_scans", "mean"),
)

# อัตราการมาทำงาน = วันที่มา / วันในข้อมูลระหว่างวันแรกถึงวันสุดท้ายของคนนั้น
def window_days(r):
    lo, hi = r.first_date.to_datetime64(), r.last_date.to_datetime64()
    return ((ALL_DATES >= lo) & (ALL_DATES <= hi)).sum()

emp["attend_rate"] = emp["days"] / emp.apply(window_days, axis=1)

# คนที่ไม่มีเวลาออกเลย (สแกนครั้งเดียวตลอด) จะถูกเติมด้วยค่ากลางของทุกคน
emp["start_std"] = emp["start_std"].fillna(0)
emp = emp.fillna(emp.median(numeric_only=True))

FEATS_A = ["start_mean", "start_std", "end_mean", "scans_mean", "attend_rate"]
Xa = StandardScaler().fit_transform(emp[FEATS_A])

sil = {k: silhouette_score(Xa, KMeans(k, n_init=20, random_state=RANDOM_STATE).fit_predict(Xa))
       for k in range(2, 7)}
print("Silhouette ของ K-Means:", {k: round(v, 3) for k, v in sil.items()})

# Silhouette มักสูงขึ้นเมื่อ k มาก เพราะเริ่มมีกลุ่มที่มีสมาชิกคนเดียว จึงกำหนดค่าเริ่มต้นไว้ที่ 4
# (เปลี่ยนเป็น K = max(sil, key=sil.get) ถ้าต้องการให้เลือกค่าที่ Silhouette สูงสุด)
K = 4
print("ใช้ k =", K)

In [ ]:
km = KMeans(K, n_init=20, random_state=RANDOM_STATE).fit(Xa)
emp["cluster_kmeans"] = km.labels_
emp["cluster_hier"] = AgglomerativeClustering(n_clusters=K, linkage="ward").fit_predict(Xa)

print("Silhouette  K-Means     :", round(silhouette_score(Xa, emp["cluster_kmeans"]), 3))
print("Silhouette  Hierarchical:", round(silhouette_score(Xa, emp["cluster_hier"]), 3))

display(emp.groupby("cluster_kmeans")[FEATS_A + ["days"]].mean().round(2))
for c, g in emp.groupby("cluster_kmeans"):
    print(f"กลุ่ม {c}:", ", ".join(g.index))

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].plot(list(sil), list(sil.values()), marker="o")
ax[0].set(xlabel="k", ylabel="Silhouette", title="เลือกจำนวนกลุ่ม (K-Means)")
sc = ax[1].scatter(emp["start_mean"], emp["end_mean"], c=emp["cluster_kmeans"], cmap="tab10", s=90)
for nm, r in emp.iterrows():
    ax[1].annotate(nm, (r["start_mean"], r["end_mean"]), fontsize=8, xytext=(4, 4), textcoords="offset points")
ax[1].set(xlabel="เวลาเข้างานเฉลี่ย (ชม.)", ylabel="เวลาเลิกงานเฉลี่ย (ชม.)", title="กลุ่มพนักงาน")
plt.tight_layout()
plt.show()

## ฟังก์ชันร่วมสำหรับส่วน B และ C

- แบ่งเทรน/ทดสอบตามเวลา
- เทรนทั้ง 6 โมเดลแล้วเทียบกัน (AUC ไม่ขึ้นกับ threshold ส่วน F1 ใช้ threshold 0.5)
- มี baseline เปรียบเทียบ ถ้าโมเดลไม่ชนะ baseline แปลว่าฟีเจอร์ยังไม่ช่วย

In [ ]:
def make_models():
    return {
        "Decision Tree": DecisionTreeClassifier(max_depth=4, min_samples_leaf=10,
                                                class_weight="balanced", random_state=RANDOM_STATE),
        "Random Forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                                class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1),
        "KNN": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15)),
        "Naive Bayes": GaussianNB(),
        "SVM (RBF)": make_pipeline(StandardScaler(),
                                   SVC(kernel="rbf", C=1.0, class_weight="balanced",
                                       probability=True, random_state=RANDOM_STATE)),
        "ANN (MLP)": make_pipeline(StandardScaler(),
                                   MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=1000,
                                                 early_stopping=True, random_state=RANDOM_STATE)),
    }

def time_split(df, frac=TRAIN_FRAC):
    dates = np.sort(df["date"].unique())
    cut = dates[int(len(dates) * frac)]
    return df[df["date"] < cut], df[df["date"] >= cut], pd.Timestamp(cut)

def evaluate(train, test, feats, target, baseline_pred, baseline_name):
    rows, fitted = [], {}
    y_te = test[target].to_numpy()
    for name, model in make_models().items():
        model.fit(train[feats], train[target])
        proba = model.predict_proba(test[feats])[:, 1]
        pred = (proba >= 0.5).astype(int)
        rows.append({"model": name,
                     "AUC": roc_auc_score(y_te, proba),
                     "PR-AUC": average_precision_score(y_te, proba),
                     "F1": f1_score(y_te, pred, zero_division=0),
                     "Accuracy": accuracy_score(y_te, pred)})
        fitted[name] = model
    rows.append({"model": baseline_name, "AUC": np.nan, "PR-AUC": np.nan,
                 "F1": f1_score(y_te, baseline_pred, zero_division=0),
                 "Accuracy": accuracy_score(y_te, baseline_pred)})
    res = pd.DataFrame(rows).set_index("model").sort_values("AUC", ascending=False)
    print(f"ทดสอบ {len(test)} แถว | สัดส่วนคลาสบวก (=1) ในชุดทดสอบ: {y_te.mean():.3f} "
          f"| Accuracy ถ้าเดาคลาสส่วนใหญ่ตลอด: {max(y_te.mean(), 1 - y_te.mean()):.3f}")
    return res.round(3), fitted

def plot_results(res, title):
    r = res.drop(index=[i for i in res.index if i.startswith("Baseline")])
    ax = r[["AUC", "F1"]].plot.bar(figsize=(9, 3.8), rot=20, title=title)
    ax.axhline(0.5, color="gray", ls="--", lw=1)   # AUC = 0.5 คือเดาสุ่ม
    ax.set_ylim(0, 1)
    plt.tight_layout()
    plt.show()

## ส่วน B – ทำนายการเลิกงานดึก

**เป้าหมาย:** `late_finish = 1` เมื่อสแกนสุดท้ายของวันตั้งแต่ `LATE_HOUR` (ค่าเริ่มต้น 19:00)
**ฟีเจอร์:** วันในสัปดาห์ + พฤติกรรมในวันทำงานก่อนหน้าของคนนั้น
(ไม่ใช้เวลาเข้างานของวันนั้นเอง เพราะค่าที่ถูกเติมด้วย KMeans คำนวณมาจากเวลาเลิกงาน จะทำให้ผลรั่ว)

In [ ]:
worked = daily.dropna(subset=["end_h"]).copy()          # ใช้เฉพาะวันที่รู้เวลาเลิกงาน
worked["late_finish"] = (worked["end_h"] >= LATE_HOUR).astype(int)
worked = worked.sort_values(["uid", "date"])
g = worked.groupby("uid")

worked["dow"] = worked["date"].dt.dayofweek
worked["prev_late"] = g["late_finish"].shift(1)
worked["roll5_late"] = g["late_finish"].transform(lambda s: s.shift(1).rolling(5, min_periods=2).mean())
worked["person_late_rate"] = g["late_finish"].transform(lambda s: s.shift(1).expanding(min_periods=3).mean())
worked["prev_start_h"] = g["start_h"].shift(1)
worked["roll5_start_h"] = g["start_h"].transform(lambda s: s.shift(1).rolling(5, min_periods=2).mean())
worked["prev_n_scans"] = g["n_scans"].shift(1)
worked["gap_days"] = g["date"].diff().dt.days

FEATS_B = ["dow", "prev_late", "roll5_late", "person_late_rate",
           "prev_start_h", "roll5_start_h", "prev_n_scans", "gap_days"]
B = worked.dropna(subset=FEATS_B).copy()
print(f"ข้อมูลส่วน B: {len(B)} แถว | สัดส่วนเลิกดึก = {B['late_finish'].mean():.3f}")

trB, teB, cutB = time_split(B)
print("เทรนก่อนวันที่", cutB.date(), "| ทดสอบตั้งแต่วันนั้น")

resB, modelsB = evaluate(trB, teB, FEATS_B, "late_finish",
                         baseline_pred=teB["prev_late"].astype(int).to_numpy(),
                         baseline_name="Baseline (เหมือนวันทำงานก่อนหน้า)")
display(resB)
plot_results(resB, "ส่วน B: ทำนายเลิกงานดึก")

In [ ]:
# ความสำคัญของฟีเจอร์ (Random Forest) และ Confusion Matrix ของโมเดลที่ AUC สูงสุด
best_b = resB.drop(index=[i for i in resB.index if i.startswith("Baseline")]).index[0]
imp = pd.Series(modelsB["Random Forest"].feature_importances_, index=FEATS_B).sort_values()

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
imp.plot.barh(ax=ax[0], title="Feature importance – Random Forest (ส่วน B)")
ConfusionMatrixDisplay.from_estimator(modelsB[best_b], teB[FEATS_B], teB["late_finish"],
                                      ax=ax[1], colorbar=False)
ax[1].set_title(f"Confusion Matrix – {best_b}")
plt.tight_layout()
plt.show()

## ส่วน C – ทำนายการขาดงาน

สร้างตาราง "พนักงาน × วัน" ตลอดช่วงที่พนักงานคนนั้นปรากฏในข้อมูล (วันแรกถึงวันสุดท้าย)
วันที่ไม่มีแถว = ขาดงาน ส่วนวันที่ระบบไม่มีข้อมูลเลยจะไม่ถูกนับเป็นขาด

**เป้าหมาย:** `absent = 1` ถ้าไม่มาในวันนั้น

In [ ]:
rows = []
for uid, gg in daily.groupby("uid"):
    lo, hi = gg["date"].min().to_datetime64(), gg["date"].max().to_datetime64()
    present = set(gg["date"])
    for d in ALL_DATES[(ALL_DATES >= lo) & (ALL_DATES <= hi)]:
        d = pd.Timestamp(d)
        rows.append((uid, d, int(d not in present)))
grid = pd.DataFrame(rows, columns=["uid", "date", "absent"]).sort_values(["uid", "date"]).reset_index(drop=True)

def streak_before(absent):
    """จำนวนวันที่มาทำงานติดต่อกันจนถึงเมื่อวาน"""
    out, run = [], 0
    for v in absent.tolist():
        out.append(run)
        run = run + 1 if v == 0 else 0
    return pd.Series(out, index=absent.index)

def days_since_absent(absent):
    """จำนวนวันนับจากวันที่ขาดครั้งล่าสุด (ก่อนวันนี้)"""
    out, last = [], None
    for i, v in enumerate(absent.tolist()):
        out.append(np.nan if last is None else i - last)
        if v == 1:
            last = i
    return pd.Series(out, index=absent.index)

g = grid.groupby("uid")["absent"]
grid["dow"] = grid["date"].dt.dayofweek
grid["prev1"] = g.shift(1)
grid["prev7"] = g.shift(7)
grid["roll7"] = g.transform(lambda s: s.shift(1).rolling(7, min_periods=3).mean())
grid["roll28"] = g.transform(lambda s: s.shift(1).rolling(28, min_periods=7).mean())
grid["person_absent_rate"] = g.transform(lambda s: s.shift(1).expanding(min_periods=7).mean())
grid["streak_present"] = g.transform(streak_before)
grid["days_since_absent"] = g.transform(days_since_absent).fillna(60)   # ยังไม่เคยขาด = ค่ามาก

FEATS_C = ["dow", "prev1", "prev7", "roll7", "roll28",
           "person_absent_rate", "streak_present", "days_since_absent"]
C = grid.dropna(subset=["prev1", "prev7", "roll7", "roll28", "person_absent_rate"]).copy()
print(f"ข้อมูลส่วน C: {len(C)} แถว | สัดส่วนขาดงาน = {C['absent'].mean():.3f}")
print("อัตราขาดงานแยกตามวัน (0=จันทร์):", C.groupby("dow")["absent"].mean().round(3).to_dict())

trC, teC, cutC = time_split(C)
print("เทรนก่อนวันที่", cutC.date(), "| ทดสอบตั้งแต่วันนั้น")

resC, modelsC = evaluate(trC, teC, FEATS_C, "absent",
                         baseline_pred=teC["prev1"].astype(int).to_numpy(),
                         baseline_name="Baseline (เหมือนเมื่อวาน)")
display(resC)
plot_results(resC, "ส่วน C: ทำนายการขาดงาน")

In [ ]:
best_c = resC.drop(index=[i for i in resC.index if i.startswith("Baseline")]).index[0]
imp = pd.Series(modelsC["Random Forest"].feature_importances_, index=FEATS_C).sort_values()

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
imp.plot.barh(ax=ax[0], title="Feature importance – Random Forest (ส่วน C)")
ConfusionMatrixDisplay.from_estimator(modelsC[best_c], teC[FEATS_C], teC["absent"],
                                      ax=ax[1], colorbar=False)
ax[1].set_title(f"Confusion Matrix – {best_c}")
plt.tight_layout()
plt.show()

## บันทึกผลเทียบโมเดล

วิธีอ่านผล: ดู **AUC** ก่อน (0.5 = เดาสุ่ม, 1.0 = สมบูรณ์แบบ) แล้วเทียบ F1 และ Accuracy กับแถว Baseline
ถ้าโมเดลไม่ชนะ Baseline อย่างชัดเจน ให้เพิ่มฟีเจอร์หรือข้อมูลก่อนนำไปใช้จริง

In [ ]:
summary = pd.concat({"B_late_finish": resB, "C_absent": resC}, names=["task"])
summary.to_csv("model_comparison.csv", encoding="utf-8-sig")
display(summary)